In [ ]:
# for google colab
# GPU check
import torch, subprocess, textwrap
print("CUDA available:", torch.cuda.is_available())
!nvidia-smi

# Clone the repo and set paths
REPO_URL = "https://github.com/MadKeyboardArtist/5703-Federated-Model.git"
REPO_DIR = "/content/5703-Federated-Model"

import os, sys
if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL}
os.chdir(REPO_DIR)
sys.path.append(REPO_DIR)   # so Python can import your modules
print("CWD:", os.getcwd())

In [17]:
# initialize model

# Per Round:
# server copies and sends the global model weights.
# client trains locally (on their local dataset).
# client returns its new weights and sample count.
# server aggregates the weights using FedAvg.
# global model is updated.
# (Optional) Evaluate global model on a held-out test set.

In [ ]:
# external libraries
import torch
import torch.nn as nn
import torch.nn.functional as F
import os
import json
import shutil

from collections import defaultdict

In [ ]:
# self-built files
from federated_multihead_model import SharedEncoders, TabularClientModel, ImageClientModel, MultiClientModel
from config import D_TABULAR, D_EMBEDDING, D_FUSION

from tabular_site_training import training as complete_tabular_training
from image_site_training   import training as complete_image_training
'''
from multi_site_training   import training_loop as multi_training_loop
'''

'\nfrom image_site_training   import training_loop as image_training_loop\nfrom multi_site_training   import training_loop as multi_training_loop\n'

In [ ]:
# reimport self-built files
import importlib
import federated_multihead_model
import tabular_site_training
importlib.reload(federated_multihead_model)
importlib.reload(tabular_site_training)

<module 'tabular_site_training' from 'd:\\USYD\\2025 S2\\5703 Capstone\\model\\tabular_site_training.py'>

In [21]:
# 1. build local model: global encoders + local heads:
def build_local_head (folder_name, model_name, modality, n_classes):
    # 1. define the complete model structure
    # solved with import
    
    # 2. assemble local model
    # 2.1 Initialize local encoders
    encoder_placeholder = SharedEncoders(
        d_tabular   = D_TABULAR, 
        d_embedding = D_EMBEDDING, 
        d_fusion    = D_FUSION
        )
    # 2.2 Load the weights
    # no need here

    # 2.3 build local model
    # modality check
    if modality == "tabular":
        local_model = TabularClientModel(shared_encoders = encoder_placeholder, n_classes = n_classes)
    elif modality == "image":
        local_model = ImageClientModel  (shared_encoders = encoder_placeholder, n_classes = n_classes)
    elif modality == "multi":
        local_model = MultiClientModel  (shared_encoders = encoder_placeholder, n_classes = n_classes)
    else:
        # report ERROR
        exit()
    
    # 2.4 save the local head weights
    os.makedirs(folder_name, exist_ok = True)
    head_path = os.path.join(folder_name, f"{model_name}.pth")
    torch.save(local_model.head.state_dict(), head_path)
    return head_path

In [ ]:
def assign_local_training_function (modality):
    if modality == "tabular":
        return complete_tabular_training
    elif modality == "image":
        return complete_image_training
    elif modality == "multi":
        pass
    else:
        # report ERROR
        return None

In [ ]:
def selective_fedavg(client_states, client_weights):
    """FedAvg per key (only keys that were returned by clients)"""

    grouped = defaultdict(list)

    for state, weight in zip(client_states, client_weights):
        for k, v in state.items():
            grouped[k].append((v, weight))

    avg_state = {}
    for k, updates in grouped.items():
        total_weight = sum(w for _, w in updates)
        avg_state[k] = sum((v * (w / total_weight)) for v, w in updates)

    return avg_state

In [ ]:
def local_training (global_state, site):
    if site["modality"] == "tabular":
        train_set_path  = site["clean_dataset_train"]
        val_set_path    = site["clean_dataset_val"]
        label_col_name  = site["label_col"]
        num_of_classes  = site["n_classes"]
        newest_head_path= site["newest_head_path"] # used for training
        
        return site["site_training"](global_state = global_state,
                                     train_set_path = train_set_path,
                                     val_set_path = val_set_path, 
                                     labelcol = label_col_name, 
                                     n_classes = num_of_classes, 
                                     newest_head_path = newest_head_path
                                     )
    
    elif site["modality"] == "image":
        train_set_path  = site["clean_dataset_train"]
        val_set_path    = site["clean_dataset_val"]
        num_of_classes  = site["n_classes"]
        newest_head_path= site["newest_head_path"] # used for training

        return site["site_training"](global_state = global_state, 
                                     train_set_path = train_set_path, 
                                     val_set_path = val_set_path, 
                                     n_classes = num_of_classes, 
                                     newest_head_path = newest_head_path
                                     )    

    elif site["modality"] == "multi":
        return None
    
    else:
        print("site modality error")
        exit()

In [25]:
def federated_training_one_round (global_state, sites):
    # 1. sever send global encoder weights to sites
    # by passing global_state
    
    ################ COME TO EACH SITES ##################
    client_updates = []
    client_heads   = []
    for site in sites:
        # full local training
        # using all site info and correct modality training function
        updated_state, sample_count, best_head_state = local_training(global_state, site) 
        
        # record the reaults
        client_updates.append((updated_state, sample_count))
        client_heads.append(best_head_state)
    ################## END in sites, back to server ##################

    # 5. Server aggregates encoder weights using FedAvg (per key, selectively)
    agg_state = selective_fedavg(
        client_states  = [s for s, _ in client_updates],
        client_weights = [w for _, w in client_updates]
    )

    # 6. Server updates global encoders with aggregated weights
    print("Finsh 1 training round")
    return agg_state, client_heads

In [ ]:
# 3. all sites preparations:
# FOR EACH SITE:
# 3.1 dataset preprocessing (remians unchanged over training)
# return: "clean_dataset_path": str, "label_col": str, "n_classes": int 
# 3.2 trai-val-test split (remians unchanged over training)
# 3.3 initialize local head (only ONCE)
# newest site head (used during training) + best site head (as outcome)
# 3.4 assign the correct local training function
# 3.5store all above info

def site_preparations (site):
    # 3.1 dataset preprocessing (remians unchanged over training)
    # return: "clean_dataset_path": str, "label_col": str, "n_classes": int
    # BEST SOLUTION: Find a way to record the preprocessing pipeline in site_info.json
    pass
        
    # placeholders:
    # site tabular_1
    site["clean_dataset"] = "tabular_dataset/diabetes_012_ready_to_model.csv"
    site["label_col"] = "Diabetes_012"
    site["n_classes"] = 2

    # 3.2 trai-val-test split (remians unchanged over training)
    pass
    # placeholders:
    site["clean_dataset_train"] = "tabular_dataset/diabetes_012_train.csv"
    site["clean_dataset_val"]   = "tabular_dataset/diabetes_012_val.csv"
    site["clean_dataset_test"]  = "tabular_dataset/diabetes_012_test.csv"

    # 3.3 initialize local head (only ONCE)
    # newest site head (used during training)
    # existing: impossible
    # NOT existing: initialize
    site["newest_head_path"] = build_local_head(
        folder_name = "newest_local_heads", 
        model_name  = site["name"], 
        modality    = site["modality"], 
        n_classes   = site["n_classes"]
        )        
    # best site head (as outcome)
    site["best_head_path"] = build_local_head(
        folder_name = "best_local_heads", 
        model_name  = site["name"], 
        modality    = site["modality"], 
        n_classes   = site["n_classes"]
        )
                
    # 3.4. assign the correct local training function
    site["site_training"] = assign_local_training_function(site["modality"])

    return site

In [42]:
# 0. (BEFORE training) clean the recorded local heads (newest)
folder_path = "newest_local_heads"
if os.path.exists(folder_path):
    shutil.rmtree(folder_path)   # deletes the folder and everything inside
    print(f"Deleted folder: {folder_path}")
else:
    print("Folder does not exist.")

# clean the recorded local heads (best)
folder_path = "best_local_heads"
if os.path.exists(folder_path):
    shutil.rmtree(folder_path)   # deletes the folder and everything inside
    print(f"Deleted folder: {folder_path}")
else:
    print("Folder does not exist.")

Deleted folder: newest_local_heads
Deleted folder: best_local_heads


In [28]:
# 1. initialize global model
global_encoders = SharedEncoders(
    d_tabular = D_TABULAR, 
    d_embedding = D_EMBEDDING, 
    d_fusion = D_FUSION
    )
global_state = global_encoders.state_dict()

In [43]:
# 2. build sites
# 2.1 run build site info (list of dict)
pass

# 2.2 import all sites (basic info)
with open("sites_info_tabular.json", "r") as f:
    sites = json.load(f)

sites

[{'name': 'tabular_1',
  'modality': 'tabular',
  'raw_dataset_path': 'tabular_dataset/diabetes_012_ready_to_model.csv'}]

In [30]:
'''
check list:

"name": "tabular_1"  : DONE
"modality": "tabular": DONE
"raw_dataset_path"   : DONE 

"clean_dataset_path": "tabular_dataset/diabetes_012_ready_to_model.csv"
"label_col": "Diabetes_012"
"n_classes": 2

"site_training" : 
"newest_head_path": 
"best_head_path":
'''

'\ncheck list:\n\n"name": "tabular_1"  : DONE\n"modality": "tabular": DONE\n"raw_dataset_path"   : DONE \n\n"clean_dataset_path": "tabular_dataset/diabetes_012_ready_to_model.csv"\n"label_col": "Diabetes_012"\n"n_classes": 2\n\n"site_training" : \n"newest_head_path": \n"best_head_path":\n'

In [44]:
# 3. all sites preparations:
# FOR EACH SITE:
# 3.1 dataset preprocessing (remians unchanged over training)
# return: "clean_dataset_path": str, "label_col": str, "n_classes": int
 
# 3.2 trai-val-test split (remians unchanged over training)

# 3.3 initialize local head (only ONCE)
# newest site head (used during training) + best site head (as outcome)

# 3.4 assign the correct local training function

# 3.5store all above info

for site in sites:
    site = site_preparations(site)

sites

[{'name': 'tabular_1',
  'modality': 'tabular',
  'raw_dataset_path': 'tabular_dataset/diabetes_012_ready_to_model.csv',
  'clean_dataset': 'tabular_dataset/diabetes_012_ready_to_model.csv',
  'label_col': 'Diabetes_012',
  'n_classes': 2,
  'clean_dataset_train': 'tabular_dataset/diabetes_012_train.csv',
  'clean_dataset_val': 'tabular_dataset/diabetes_012_val.csv',
  'clean_dataset_test': 'tabular_dataset/diabetes_012_test.csv',
  'newest_head_path': 'newest_local_heads\\tabular_1.pth',
  'best_head_path': 'best_local_heads\\tabular_1.pth',
  'site_training': <function tabular_site_training.training(global_state, train_set_path, val_set_path, labelcol, n_classes, newest_head_path)>}]

In [45]:
# 4. operate federated training loop
training_round = 5
best_state = global_state

for i in range(training_round):
    # 1. one federated training round -> ALL sites FULLY trained ONCE
    new_global_state, new_best_heads = federated_training_one_round(global_state, sites)
    
    # 2. Evaluation on val set
    # using the new_global_state and all the new best_heads

    # 3. record the new model (global + all heads if they are the best)
    
    # 4. start the next round
    global_state = new_global_state

 [best updated] 0.8363
finish site training
Finsh 1 training round
 [best updated] 0.8273
finish site training
Finsh 1 training round
 [best updated] 0.8238
finish site training
Finsh 1 training round
 [best updated] 0.8161
 [best updated] 0.8213
finish site training
Finsh 1 training round
 [best updated] 0.8033
 [best updated] 0.8100
 [best updated] 0.8106
 [best updated] 0.8106
finish site training
Finsh 1 training round


In [ ]:
# 4. final evaluation